In [8]:
# laod the result sheets first
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "outputs").is_dir())

REPORTS = {
    "rank_sweep": ROOT / "outputs/pipeline_runs/moe_study_rank_sweep_pipeline_report/rank_pipeline_report.xlsx",
    "expert_sweep": ROOT / "outputs/pipeline_runs/moe_study_expert_sweep_pipeline_report/expert_pipeline_report.xlsx",
}

In [9]:
# in this cell we prepare the df we use for analysis
import re
import pandas as pd
from IPython.display import display, HTML

def parse_configured_rank(run_id):
    run_id = str(run_id)

    total_rank = re.search(r"_tr(\d+)_", run_id)
    experts = re.search(r"_e(\d+)(?:_|$)", run_id)

    # moe approaches have total rank and experts, to derive the configured rank, we divide the total rank by the number of experts
    if total_rank and experts:
        return int(total_rank.group(1)) / int(experts.group(1))

    rank = re.search(r"_r(\d+)(?:_|$)", run_id)
    return int(rank.group(1)) if rank else pd.NA


def parse_component_count(run_id):
    # we dont always have experts, e.g hydralora and mtllora use "_heads" or number of projections "_nup"
    patterns = [
        r"_e(\d+)(?:_|$)",
        r"_heads(\d+)$",
        r"_nup(\d+)$",
        r"_b(\d+)$",
    ]

    for pattern in patterns:
        match = re.search(pattern, str(run_id))
        if match:
            return int(match.group(1))

    return pd.NA


def clean_benchmark_data(data):
    # 1. keep only the primary metric for each benchmark, f1 scores are not considered
    primary_mask = (
        ((data.benchmark == "Belebele") & (data.metric == "acc_norm"))
        | ((data.benchmark == "FLORES") & data.metric.astype(str).str.startswith("bits_per_byte "))
        | ((data.benchmark == "SIB200") & (data.metric == "acc"))
        | ((data.benchmark == "Taxi1500") & (data.metric == "acc"))
    )
    data = data[primary_mask].copy()

    # 2. exclude baseline, we have some baseline resutls in the excel sheert, but here we focus on #experts and lora-rank
    is_baseline = (
        data["is_baseline"]
        .fillna(False)
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes"])
    )
    data = data[~is_baseline].copy()
    data = data[data.kind.astype(str).ne("cola")].copy() # exclude CoLA because it did not recover from a loss spike, and failed training, the results would skew the analysis
    data["value"] = pd.to_numeric(data["value"], errors="coerce")
    data = data.dropna(subset=["value"]).copy()

    # 3. keep both FLORES directions separate
    direction = (
        data.metric.astype(str)
        .str.extract(r"(en->x|x->en)", expand=False)
        .fillna("")
    )

    data["score_family"] = data["benchmark"]
    flores_mask = data.benchmark.eq("FLORES")
    data.loc[flores_mask, "score_family"] = (
        "FLORES " + direction.loc[flores_mask]
    )

    # 4. derive lora rank and expert / head / projection count from the run ID
    data["configured_rank"] = data.run_id.map(parse_configured_rank)
    data["expert / head / projection"] = data.run_id.map(parse_component_count)
    data.drop(columns=["is_baseline"], inplace=True)

    return data


# Now I create the two separate cleaned DataFrames directly
rank_data = clean_benchmark_data(
    pd.read_excel(REPORTS["rank_sweep"], sheet_name="benchmark_raw")
)
rank_data = rank_data.dropna(subset=["configured_rank"]).copy()
rank_data["configured_rank"] = rank_data.configured_rank.astype(int)
rank_data.drop(columns=["expert / head / projection"], inplace=True)


expert_data = clean_benchmark_data(
    pd.read_excel(REPORTS["expert_sweep"], sheet_name="benchmark_raw")
)
expert_data = expert_data.dropna(subset=["expert / head / projection"]).copy()
expert_data["expert / head / projection"] = expert_data["expert / head / projection"].astype(int)
expert_data.drop(columns=["configured_rank"], inplace=True)


print("This includes all benchmark results accross all benchmarks, approaches, languages and configured ranks")
print("\n Note: movlora is not included since it does not use lora ranks, Cola is excluded")
display(rank_data.head(10))
# this shows the full scrollable tables
#display(HTML(f""" 
#<div style="max-height:500px; overflow-y:auto;">
#{rank_data.to_html()}
#</div>
#"""))

print("This includes all benchmark results accross all benchmarks, approaches, languages and number of experts / heads / projections")
print("\n Note: CoLA is exluded here")
display(expert_data.head(10))
#display(HTML(f"""
#<div style="max-height:500px; overflow-y:auto;">
#{expert_data.to_html()}
#</div>
#"""))

This includes all benchmark results accross all benchmarks, approaches, languages and configured ranks

 Note: movlora is not included since it does not use lora ranks, Cola is excluded


,run_id,kind,benchmark,task,language,resource_availability,metric,better,value,score_family,configured_rank
9,hmora_tr1024_e8_k2,hmora,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.308889,Belebele,128
10,hmora_tr128_e8_k2,hmora,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.387778,Belebele,16
11,hmora_tr16_e8_k2,hmora,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.566667,Belebele,2
12,hmora_tr256_e8_k2,hmora,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.353333,Belebele,32
13,hmora_tr32_e8_k2,hmora,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.543333,Belebele,4
14,hmora_tr512_e8_k2,hmora,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.223333,Belebele,64
15,hmora_tr64_e8_k2,hmora,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.542222,Belebele,8
16,hydralora_r128,hydralora,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.506667,Belebele,128
17,hydralora_r16,hydralora,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.575556,Belebele,16
18,hydralora_r2,hydralora,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.587778,Belebele,2


This includes all benchmark results accross all benchmarks, approaches, languages and number of experts / heads / projections

 Note: CoLA is exluded here


,run_id,kind,benchmark,task,language,resource_availability,metric,better,value,score_family,expert / head / projection
0,adamoe_1_r2_e16_k1,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.600000,Belebele,16
1,adamoe_1_r2_e16_k2,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.588889,Belebele,16
2,adamoe_1_r2_e16_k4,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.594444,Belebele,16
3,adamoe_1_r2_e16_k8,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.597778,Belebele,16
4,adamoe_1_r2_e2_k1,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.577778,Belebele,2
5,adamoe_1_r2_e32_k1,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.593333,Belebele,32
6,adamoe_1_r2_e32_k16,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.601111,Belebele,32
7,adamoe_1_r2_e32_k2,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.580000,Belebele,32
8,adamoe_1_r2_e32_k4,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.614444,Belebele,32
9,adamoe_1_r2_e32_k8,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.600000,Belebele,32


In [10]:
# now we perform friedman test to derive best lora rank
import pingouin as pg
from statsmodels.stats.multitest import multipletests

# first we cehck how many rows we have for each rank
rank_coverage = rank_data.groupby("configured_rank").agg(
    rows=("value", "size"),
    configurations=("run_id", "nunique"),
    approaches=("kind", "nunique"),
)
display(rank_coverage)

# we will only inlcude the ranks 2, 4, 8, 16, 32, 64 because friedmann test requires the same number of rows for each unit that will be measured
balanced_ranks = rank_coverage.index[rank_coverage.approaches.eq(rank_coverage.approaches.max())].tolist() # here we select ranks 2 - 64
print("Compared ranks:", balanced_ranks)
analysis = rank_data[rank_data.configured_rank.isin(balanced_ranks)].copy()

# SIB / Belebele / TAXI use accuracy (acc or acc_norm) where higher is beeter
# flores uses perplexity where lower is better
# to rank all rows with higher is better, we can simply invert the flores perplexity resutls. e.g. 1.42 becomes -1.42, thus we can rank each row safely using hihger is better
analysis["score"] = analysis["value"].where(
    analysis.better.ne("lower"),
    -analysis["value"],
)

results = []
for benchmark in ["Belebele", "FLORES", "SIB200", "Taxi1500"]:
    data = analysis[analysis.benchmark.eq(benchmark)]

    # Blocks = evaluation unit × approach; treatments = configured ranks
    pivot = data.pivot(
        index=["score_family", "language", "kind"],
        columns="configured_rank",
        values="score",
    ).reindex(columns=balanced_ranks)
    
    # we need to be very careful here. If we instead take the mean of all approaches per benchmark it changes the outcome of the friedmann test. The seond place then is 4 instead of 8
    # pivot = (
    #     data.groupby(
    #         ["score_family", "task", "language", "metric", "configured_rank"]
    #     )["score"]
    #     .mean()
    #     .unstack("configured_rank")
    #     .reindex(columns=balanced_ranks)
    # )
    
    if benchmark == "Belebele":  # first loop iteration only
      print("i thinkg its helpful here to look at thsi table to understand what exactly is given into the friedmann test, uncomment to see")
      print("Pivot table for:", benchmark)
      #display(pivot_0.head(10))
      display(pivot.head(10))

    unit_ranks = pivot.rank(axis=1, ascending=False)
    # if benchmark == "Belebele":  # first loop iteration only
    #   print("i thinkg its helpful here to look at thsi table to understand what exactly is given into the friedmann test, uncomment to see")
    #   print("Unit ranks for:", benchmark)
    #   display(unit_ranks)
    average_ranks = unit_ranks.mean()
    # if benchmark == "Belebele":  # first loop iteration only
    #   print("i thinkg its helpful here to look at thsi table to understand what exactly is given into the friedmann test, uncomment to see")
    #   print("Average ranks for:", benchmark)
    #   display(average_ranks)

    # now all the rows for one benchmark into friedmann test
    friedman = pg.friedman(data=pivot)
    statistic = friedman["Q"].iat[0]
    p_value = friedman["p_unc"].iat[0]
    kendalls_w = friedman["W"].iat[0]
    best_two = average_ranks.nsmallest(2).index.tolist()

    results.append({
        "benchmark": benchmark,
        "best_rank": average_ranks.idxmin(),
        "best_two_ranks": best_two[1],
        "friedman_statistic": statistic,
        "p_raw": p_value,
        "kendalls_w": kendalls_w,
        **{
            f"avg_rank_or_lora_rank{rank}": average_ranks[rank]
            for rank in balanced_ranks
        },
    })

results = pd.DataFrame(results)
results["p_holm"] = multipletests(results.p_raw, method="holm")[1]

display(results.round(7))

,rows,configurations,approaches
configured_rank,,,
2,252,6,6
4,252,6,6
8,252,6,6
16,252,6,6
32,252,6,6
64,252,6,6
128,210,5,5
256,126,3,3
512,42,1,1


Compared ranks: [2, 4, 8, 16, 32, 64]
i thinkg its helpful here to look at thsi table to understand what exactly is given into the friedmann test, uncomment to see
Pivot table for: Belebele


configured_rank                        2         4         8         16  \
score_family language kind                                                
Belebele     arb_Arab hmora      0.566667  0.543333  0.542222  0.387778   
                      hydralora  0.587778  0.592222  0.578889  0.575556   
                      lora       0.585556  0.594444  0.587778  0.596667   
                      moelora    0.594444  0.600000  0.595556  0.616667   
                      mola       0.602222  0.596667  0.592222  0.588889   
                      mtllora    0.576667  0.592222  0.585556  0.582222   
             arb_Latn hmora      0.290000  0.263333  0.276667  0.244444   
                      hydralora  0.298889  0.298889  0.282222  0.285556   
                      lora       0.275556  0.266667  0.280000  0.257778   
                      moelora    0.276667  0.274444  0.278889  0.280000   

configured_rank                        32        64  
score_family language kind                           
Belebele     arb_Arab hmora      0.353333  0.223333  
                      hydralora  0.581111  0.565556  
                      lora       0.580000  0.563333  
                      moelora    0.560000  0.588889  
                      mola       0.591111  0.575556  
                      mtllora    0.564444  0.577778  
             arb_Latn hmora      0.238889  0.235556  
                      hydralora  0.292222  0.268889  
                      lora       0.257778  0.271111  
                      moelora    0.268889  0.281111

,benchmark,best_rank,best_two_ranks,friedman_statistic,p_raw,kendalls_w,avg_rank_or_lora_rank2,avg_rank_or_lora_rank4,avg_rank_or_lora_rank8,avg_rank_or_lora_rank16,avg_rank_or_lora_rank32,avg_rank_or_lora_rank64,p_holm
0,Belebele,8,2,106.238655,0.000000,0.393477,2.759259,2.851852,2.407407,3.009259,4.944444,5.027778,0.000000
1,FLORES,2,8,341.433862,0.000000,0.632285,2.000000,2.370370,2.361111,3.851852,4.833333,5.583333,0.000000
2,SIB200,2,8,13.603579,0.018334,0.050384,2.888889,3.407407,3.324074,3.629630,3.601852,4.148148,0.018334
3,Taxi1500,2,4,74.522613,0.000000,0.414015,2.305556,2.625000,2.791667,3.458333,4.597222,5.222222,0.000000


In [11]:
overall_ranks = pd.Series({
    rank: results[f"avg_rank_or_lora_rank{rank}"].mean()
    for rank in balanced_ranks
}).sort_values()

display(overall_ranks)

best_overall = overall_ranks.index[0]
second_best_overall = overall_ranks.index[1]

print("Best overall:", best_overall)
print("Second best overall:", second_best_overall)

2     2.488426
8     2.721065
4     2.813657
16    3.487269
32    4.494213
64    4.995370
dtype: float64

Best overall: 2
Second best overall: 8


## Now for experts

In [12]:
# Now perform Friedman test to derive the best expert count
import pingouin as pg
from statsmodels.stats.multitest import multipletests

TARGET_COL = "expert / head / projection"
UNIT_COLS = ["score_family", "task", "language", "metric"]

# Check expert-count coverage
expert_coverage = expert_data.groupby(TARGET_COL).agg(
    rows=("value", "size"),
    configurations=("run_id", "nunique"),
    approaches=("kind", "nunique"),
)
display(expert_coverage)
print("We first see that the number of rows we have here for each expert count it highly unbalanced. This also means that we cannot use the mean of all approaches per benchmark, because some approaches are missing for some expert counts. We will have to filter out the approaches that are not available for all expert counts, and then perform the friedman test as before, since it requires same amount of rows for each expert count")

configurations_per_kind = ( # lets first look into why the number of rows is unbalances
    expert_data.groupby(
        ["expert / head / projection", "kind"]
    )["run_id"]
    .nunique()
    .unstack(fill_value=0)
)
display(configurations_per_kind)

topk_coverage = expert_data[[TARGET_COL, "kind", "run_id"]].drop_duplicates()
topk_coverage["top_k"] = (topk_coverage.run_id.astype(str).str.extract(r"_k(\d+)(?:_|$)", expand=False))
topk_table = (
    topk_coverage.groupby([TARGET_COL, "kind"])["top_k"]
    .agg(lambda x: ", ".join(
        f"k{v}" for v in sorted(x.dropna().unique(), key=int)
    ) if x.notna().any() else "n/a")
    .unstack(fill_value="missing")
)
display(topk_table)
print("We see that the number of rows per expert count is unbalanced because we have multiple top-k values for configurations with more experts, which makes sense and is expected.")


expert_counts = sorted(expert_data[TARGET_COL].unique())
# Keep only approaches available at every expert count
common_kinds = sorted(set.intersection(*[
    set(expert_data.loc[expert_data[TARGET_COL].eq(count), "kind"])
    for count in expert_counts
]))
print("Compared expert counts:", expert_counts)
print("Common approaches:", common_kinds)
expert_analysis = expert_data[expert_data.kind.isin(common_kinds)].copy()
# make higher always better
expert_analysis["score"] = expert_analysis["value"].where(
    expert_analysis.better.ne("lower"),
    -expert_analysis["value"],
)

# higher expert counts contain multiple top-k configurations per approach
# we average these configurations so that every expert count has one score per approach
# e.g hmore with expert count 8 has 3 runs, with topk of 1,2,4. We average these 3 runs per benchmark to get a average score for hmore with 8 experts
# one score per approach and expert count.
approach_scores = (
    expert_analysis.groupby(
        ["benchmark"]+ UNIT_COLS + ["kind", TARGET_COL],
        as_index=False,
    )
    .agg(
        score=("score", "mean"),
        configurations_averaged=("run_id", "nunique"),
    )
)

# Confirm that the collapsed data is balanced
collapsed_coverage = approach_scores.groupby(TARGET_COL).agg(
    rows=("score", "size"),
    approaches=("kind", "nunique"),
)
display(collapsed_coverage)
print("We have collapsed the approaches now by averaging, this should be reviewed if this is fine. \n Thus we get the same number of rows for each expert count and can give it into Friedman.")
display(approach_scores.head(10))


expert_results = []

for benchmark in ["Belebele", "FLORES", "SIB200", "Taxi1500"]:
    data = approach_scores[
        approach_scores["benchmark"].eq(benchmark)
    ]

    # Blocks = evaluation unit × approach
    # Treatments = expert counts
    pivot = data.pivot(
        index=UNIT_COLS + ["kind"],
        columns=TARGET_COL,
        values="score",
    ).reindex(columns=expert_counts)

    if pivot.isna().any().any():
        raise ValueError(
            f"Incomplete expert-count panel for {benchmark}"
        )

    if benchmark == "Belebele":
        print("Pivot table for:", benchmark)
        display(pivot.head(10))

    unit_ranks = pivot.rank(axis=1, ascending=False)
    average_ranks = unit_ranks.mean()
    best_two = average_ranks.nsmallest(2).index.tolist()

    friedman = pg.friedman(data=pivot)

    expert_results.append({
        "benchmark": benchmark,
        "best_expert_count": best_two[0],
        "second_best_expert_count": best_two[1],
        "friedman_statistic": friedman["Q"].iat[0],
        "p_raw": friedman["p_unc"].iat[0],
        "kendalls_w": friedman["W"].iat[0],
        **{
            f"avg_rank_expert_count_{count}": average_ranks[count]
            for count in expert_counts
        },
    })

expert_results = pd.DataFrame(expert_results)

expert_results["p_holm"] = multipletests(
    expert_results.p_raw,
    method="holm",
)[1]

display(expert_results.round(7))


overall_expert_ranks = pd.Series({
    count: expert_results[f"avg_rank_expert_count_{count}"].mean()
    for count in expert_counts
}).sort_values()

display(overall_expert_ranks)

print("Best expert count overall:", overall_expert_ranks.index[0])
print("Second best expert count overall:", overall_expert_ranks.index[1])

,rows,configurations,approaches
expert / head / projection,,,
2,294,7,7
4,420,10,7
8,546,13,7
16,672,16,7
32,714,17,6


We first see that the number of rows we have here for each expert count it highly unbalanced. This also means that we cannot use the mean of all approaches per benchmark, because some approaches are missing for some expert counts. We will have to filter out the approaches that are not available for all expert counts, and then perform the friedman test as before, since it requires same amount of rows for each expert count


kind,adamole,hmora,hydralora,moelora,mola,movlora,mtllora
expert / head / projection,,,,,,,
2,1,1,1,1,1,1,1
4,1,2,1,2,2,1,1
8,1,3,1,3,3,1,1
16,1,4,1,4,4,1,1
32,0,5,1,4,5,1,1


kind,adamole,hmora,hydralora,moelora,mola,movlora,mtllora
expert / head / projection,,,,,,,
2,n/a,k1,n/a,k1,k1,n/a,n/a
4,n/a,"k1, k2",n/a,"k1, k2","k1, k2",n/a,n/a
8,n/a,"k1, k2, k4",n/a,"k1, k2, k4","k1, k2, k4",n/a,n/a
16,n/a,"k1, k2, k4, k8",n/a,"k1, k2, k4, k8","k1, k2, k4, k8",n/a,n/a
32,missing,"k1, k2, k4, k8, k16",n/a,"k1, k2, k4, k8","k1, k2, k4, k8, k16",n/a,n/a


We see that the number of rows per expert count is unbalanced because we have multiple top-k values for configurations with more experts, which makes sense and is expected.
Compared expert counts: [2, 4, 8, 16, 32]
Common approaches: ['hmora', 'hydralora', 'moelora', 'mola', 'movlora', 'mtllora']


,rows,approaches
expert / head / projection,,
2,252,6
4,252,6
8,252,6
16,252,6
32,252,6


... (3 outputs left out)

expert / head / projection                                        2   \
score_family task              language metric   kind                  
Belebele     belebele_arb_Arab arb_Arab acc_norm hmora      0.543333   
                                                 hydralora  0.588889   
                                                 moelora    0.583333   
                                                 mola       0.577778   
                                                 movlora    0.585556   
                                                 mtllora    0.596667   
             belebele_arb_Latn arb_Latn acc_norm hmora      0.287778   
                                                 hydralora  0.270000   
                                                 moelora    0.273333   
                                                 mola       0.290000   

expert / head / projection                                        4   \
score_family task              language metric   kind                  
Belebele     belebele_arb_Arab arb_Arab acc_norm hmora      0.454444   
                                                 hydralora  0.586667   
                                                 moelora    0.595000   
                                                 mola       0.593333   
                                                 movlora    0.584444   
                                                 mtllora    0.588889   
             belebele_arb_Latn arb_Latn acc_norm hmora      0.246667   
                                                 hydralora  0.281111   
                                                 moelora    0.283333   
                                                 mola       0.280556   

expert / head / projection                                        8   \
score_family task              language metric   kind                  
Belebele     belebele_arb_Arab arb_Arab acc_norm hmora      0.442222   
                                                 hydralora  0.584444   
                                                 moelora    0.586667   
                                                 mola       0.595556   
                                                 movlora    0.577778   
                                                 mtllora    0.583333   
             belebele_arb_Latn arb_Latn acc_norm hmora      0.252222   
                                                 hydralora  0.270000   
                                                 moelora    0.286667   
                                                 mola       0.288519   

expert / head / projection                                        16        32  
score_family task              language metric   kind                           
Belebele     belebele_arb_Arab arb_Arab acc_norm hmora      0.441389  0.551556  
                                                 hydralora  0.598889  0.595556  
                                                 moelora    0.603056  0.586389  
                                                 mola       0.595278  0.597778  
                                                 movlora    0.581111  0.585556  
                                                 mtllora    0.591111  0.582222  
             belebele_arb_Latn arb_Latn acc_norm hmora      0.237222  0.296667  
                                                 hydralora  0.292222  0.286667  
                                                 moelora    0.273611  0.276111  
                                                 mola       0.286944  0.289333

,benchmark,best_expert_count,second_best_expert_count,friedman_statistic,p_raw,kendalls_w,avg_rank_expert_count_2,avg_rank_expert_count_4,avg_rank_expert_count_8,avg_rank_expert_count_16,avg_rank_expert_count_32,p_holm
0,Belebele,32,4,20.872659,0.000336,0.096633,3.157407,2.796296,3.759259,2.814815,2.472222,0.000671
1,FLORES,32,4,62.985185,0.000000,0.145799,3.592593,2.888889,2.935185,3.500000,2.083333,0.000000
2,SIB200,16,4,28.469428,0.000010,0.131803,3.398148,2.666667,3.268518,2.185185,3.481482,0.000030
3,Taxi1500,32,16,20.539130,0.000391,0.142633,3.680556,3.000000,3.472222,2.513889,2.333333,0.000671


32    2.592593
16    2.753472
4     2.837963
8     3.358796
2     3.457176
dtype: float64

Best expert count overall: 32
Second best expert count overall: 16


## now we decide which specific approaches to use for longer follow up training

In [13]:
SELECTED_RANKS = [2, 8]
BENCHMARKS = ["Belebele", "FLORES", "SIB200", "Taxi1500"]
UNIT_COLS = ["score_family", "task", "language", "metric"]

rank_config_results = {}

for rank in SELECTED_RANKS:
    data = rank_data[rank_data.configured_rank.eq(rank)].copy()

    # higher must always mean better
    data["score"] = data["value"].where(
        data["better"].ne("lower"),
        -data["value"],
    )

    # Compare concrete configurations within this configured rank
    pivot = data.pivot(
        index=UNIT_COLS,
        columns="run_id",
        values="score",
    )

    # rank configurations inside every evaluation unit
    unit_ranks = pivot.rank(
        axis=1,
        ascending=False,
        method="average",
    )
    family_ranks = unit_ranks.groupby(level="score_family").mean()

    # Combine FLORES directions after ranking
    family_ranks.loc["FLORES"] = family_ranks.loc[
        ["FLORES en->x", "FLORES x->en"]
    ].mean()
    benchmark_ranks = family_ranks.loc[BENCHMARKS]
    result = benchmark_ranks.T # transpose
    result["macro_avg_rank"] = benchmark_ranks.mean()

    approach = (
        data.drop_duplicates("run_id")
        .set_index("run_id")["kind"]
    )
    result.insert(0, "approach", approach.reindex(result.index))

    result = result.sort_values("macro_avg_rank")
    rank_config_results[rank] = result

    print(f"\nConfigured rank: {rank}")
    display(result)
    
    # DEBUG: uncomment to inspect every intermediate step for FLORES/rank 2
    # if rank == 2:
    #     flores_units = (
    #         pivot.index
    #         .get_level_values("score_family")
    #         .str.startswith("FLORES")
    #     )
    
    #     print("1. Original FLORES rows")
    #     display(
    #         data[data.benchmark.eq("FLORES")][
    #             ["run_id", "score_family", "task",
    #              "language", "metric", "value", "score"]
    #         ]
    #     )
    
    #     print("2. FLORES values given to the ranking")
    #     display(pivot.loc[flores_units])
    
    #     print("3. Ranks inside every FLORES evaluation unit")
    #     display(unit_ranks.loc[flores_units])
    
    #     print("4. Average rank for each FLORES direction")
    #     display(
    #         family_ranks.loc[
    #             ["FLORES en->x", "FLORES x->en"]
    #         ]
    #     )
    
    #     print("5. Combined FLORES rank")
    #     display(family_ranks.loc[["FLORES"]])


Configured rank: 2


score_family,approach,Belebele,FLORES,SIB200,Taxi1500,macro_avg_rank
run_id,,,,,,
moelora_tr16_e8_k2,moelora,1.888889,3.388889,2.444444,1.166667,2.222222
hydralora_r2,hydralora,3.277778,3.944444,1.944444,2.916667,3.020833
mola_tr16_e8_k6,mola,2.055556,1.944444,3.777778,4.333333,3.027778
hmora_tr16_e8_k2,hmora,5.222222,5.500000,2.000000,2.666667,3.847222
mtllora_r2,mtllora,4.888889,2.555556,5.611111,4.583333,4.409722
lora_r2,lora,3.666667,3.666667,5.222222,5.333333,4.472222



Configured rank: 8


score_family,approach,Belebele,FLORES,SIB200,Taxi1500,macro_avg_rank
run_id,,,,,,
moelora_tr64_e8_k2,moelora,1.833333,3.666667,1.222222,3.250000,2.493056
mtllora_r8,mtllora,3.500000,3.444444,2.444444,1.083333,2.618056
mola_tr64_e8_k6,mola,2.888889,3.333333,3.000000,3.666667,3.222222
hydralora_r8,hydralora,3.944444,2.111111,3.777778,3.333333,3.291667
lora_r8,lora,2.944444,2.444444,5.222222,5.083333,3.923611
hmora_tr64_e8_k2,hmora,5.888889,6.000000,5.333333,4.583333,5.451389


based on that we select and promote
 - moelora_tr16_e8_k2
 - hydralora_r2
 - mola_tr16_e8_k6

as the best configuratiosn with a configured rank of 2
 
---
and we select
 - moelora_tr64_e8_k2
 - mtllora_r8  
 - mola_tr64_e8_k6  

as the best configuraitons with a configured rank of 8

In [14]:
# rank concrete configurations within selected component counts
SELECTED_COUNTS = [32, 16]
BENCHMARKS = ["Belebele", "FLORES", "SIB200", "Taxi1500"]
UNIT_COLS = ["score_family", "task", "language", "metric"]

config_rankings = {}

for count in SELECTED_COUNTS:
    data = expert_data[expert_data[TARGET_COL].eq(count)].copy()

    # higher must always mean better
    data["score"] = data["value"].where(
        data["better"].ne("lower"),
        -data["value"],
    )

    # Rows = evaluation units, columns = concrete configurations
    pivot = data.pivot(
        index=UNIT_COLS,
        columns="run_id",
        values="score",
    )

    # rank concrete configurations inside every evaluation unit
    unit_ranks = pivot.rank(
        axis=1,
        ascending=False,
        method="average",
    )

    # average ranks separately for each benchmark/direction
    family_ranks = unit_ranks.groupby(level="score_family").mean()
    family_ranks.loc["FLORES"] = family_ranks.loc[ # Combine the two FLORES directions after ranking, note this is average over the ranks not over the scores
        ["FLORES en->x", "FLORES x->en"]
    ].mean()
    benchmark_ranks = family_ranks.loc[BENCHMARKS]

    # One row per concrete configuration
    result = benchmark_ranks.T
    result["macro_avg_rank"] = benchmark_ranks.mean()

    approach = (
        data.drop_duplicates("run_id")
        .set_index("run_id")["kind"]
    )
    result.insert(0, "approach", approach.reindex(result.index))

    result = result.sort_values("macro_avg_rank")
    config_rankings[count] = result

    print(f"\nComponent count: {count}")
    display(result)


Component count: 32


score_family,approach,Belebele,FLORES,SIB200,Taxi1500,macro_avg_rank
run_id,,,,,,
moelora_r2_e32_k8,moelora,5.055556,5.944444,4.500000,2.583333,4.520833
adamoe_1_r2_e32_k4,mola,5.277778,5.388889,4.666667,7.000000,5.583333
moelora_r2_e32_k4,moelora,6.888889,3.944444,7.222222,5.916667,5.993056
adamoe_1_r2_e32_k2,mola,10.000000,4.555556,6.888889,3.000000,6.111111
adamoe_1_r2_e32_k16,mola,4.000000,10.555556,5.777778,7.916667,7.062500
adamoe_1_r2_e32_k8,mola,3.222222,5.444444,5.444444,15.083333,7.298611
hydralora_r2_heads32,hydralora,5.555556,3.722222,9.000000,11.166667,7.361111
adamoe_1_r2_e32_k1,mola,7.833333,9.055556,4.055556,14.333333,8.819444
moelora_r2_e32_k2,moelora,12.833333,7.111111,9.055556,6.666667,8.916667



Component count: 16


score_family,approach,Belebele,FLORES,SIB200,Taxi1500,macro_avg_rank
run_id,,,,,,
moelora_r2_e16_k8,moelora,4.000000,4.222222,4.777778,2.000000,3.750000
adamoe_1_r2_e16_k1,mola,3.777778,9.111111,4.944444,3.833333,5.416667
adamoe_1_r2_e16_k8,mola,8.000000,3.222222,6.388889,5.000000,5.652778
adamoe_1_r2_e16_k2,mola,7.166667,3.555556,4.888889,8.000000,5.902778
adamoe_1_r2_e16_k4,mola,7.944444,4.388889,5.000000,7.500000,6.208333
moelora_r2_e16_k2,moelora,3.222222,6.333333,11.833333,4.083333,6.368056
hydralora_r2_heads16,hydralora,6.611111,4.611111,7.111111,8.583333,6.729167
moelora_r2_e16_k4,moelora,5.611111,5.722222,7.222222,8.916667,6.868056
mtllora_r2_nup16,mtllora,8.500000,6.166667,8.555556,4.250000,6.868056


based on that we select and promote
 - moelora_r2_e32_k8
 - adamoe_1_r2_e32_k4
 - moelora_r2_e32_k4

as the best configuratiosn with a compoment count of 32
 
---
and we select
 - moelora_r2_e16_k8
 - adamoe_1_r2_e16_k1  
 - adamoe_1_r2_e16_k8  

as the best configuraitons with a component count of 16

## Mixing up selected configurations

selected for rank 2:
 - moelora_tr16_e8_k2
 - hydralora_r2
 - mola_tr16_e8_k6

selected for rank 8
 - moelora_tr64_e8_k2
 - mtllora_r8  
 - mola_tr64_e8_k6  

selected for expert count 32
 - moelora_r2_e32_k8
 - adamoe_1_r2_e32_k4
 - moelora_r2_e32_k4

selected for expert count 16
 - moelora_r2_e16_k8
 - adamoe_1_r2_e16_k1  
 - adamoe_1_r2_e16_k8 

---
note that adamoe_1 = mola, the naming or run ids was not consistent

List of mixed follow up approaches based on that: 
- first, take all selected expert configs and use with rank 8 and 2
    - moelora_tr64_e32_k8    (rank2)
    - moelora_tr256_e32_k8   (rank8)
    - adamoe_1_tr64_e32_k4   (rank2)
    - adamoe_1_tr256_e32_k4  (rank8)
    - moelora_tr64_e32_k4    (rank2)
    - moelora_tr256_e32_k4   (rank8)

    - moelora_tr32_e16_k8    (rank2)
    - moelora_tr128_e16_k8   (rank8)
    - adamoe_1_tr32_e16_k1   (rank2)
    - adamoe_1_tr128_e16_k1  (rank8)
    - adamoe_1_tr32_e16_k8   (rank2)
    - adamoe_1_tr128_e16_k8  (rank8)
- now take all the selected rank configs:
    - moelora_tr64_e32_k4  (1. rank2 config mixed with 32 epxerts and topk4)
    - moelora_tr64_e32_k8  (1. rank2 config mixed with 32 epxerts and topk8)
    - moelora_tr32_e16_k8  (1. rank2 config mixed with 16 epxerts and topk8)

    - hydralora_tr64_heads_32 (2. rank2 config mixed with 32 experts, no topk)
    - hydralora_tr32_heads_16 (2. rank2 config mixed with 16 experts, no topk)

    - mola_tr64_e32_k4  (3rd rank2 config mixed with 32 experts and topk4)
    - mola_tr32_e16_k1  (3rd rank2 config mixed with 16 experts and topk1)
    - mola_tr32_e16_k8  (3rd rank2 config mixed with 16 experts and topk8)
    

    - moelora_tr256_e32_k8 (1st rank8 config mixed with 32 experts and topk8)
    - moelora_tr256_e32_k4 (1st rank8 config mixed with 32 experts and topk4)
    - moelora_tr128_e16_k8 (1st rank8 config mixed with 16 experts and topk8)


    - mtllora_tr_256_nump_32 (2nd rank8 config mixed with 32 experts)
    - mtllora_tr_128_nump_16 (2nd rank8 config mixed with 16 experts)

    - mola_tr256_e32_k4 (3rd rank8 config mixed with 32 experts and topk4)
    - mola_tr128_e16_k1 (3rd rank8 config mixed with 16 experts and topk1)
    - mola_tr128_e16_k8 (3rd rank8 config mixed with 16 experts and topk8)





for mixing best rank configs with best experts configs i needed to establish this rule:
- Joint the selected rank configurations and selected expert configurations by approach family. Keep the selected configured rank and use every (expert count, top-k) combination that appears among that families selected expert family configurations.

deduplicated the final list is:
- moelora_tr64_e32_k8   
- moelora_tr256_e32_k8  
- adamoe_1_tr64_e32_k4  
- adamoe_1_tr256_e32_k4 
- moelora_tr64_e32_k4   
- moelora_tr256_e32_k4  
- moelora_tr32_e16_k8   
- moelora_tr128_e16_k8  
- adamoe_1_tr32_e16_k1  
- adamoe_1_tr128_e16_k1 
- adamoe_1_tr32_e16_k8  
- adamoe_1_tr128_e16_k8 
- hydralora_tr64_heads_32
- hydralora_tr32_heads_16
- mtllora_tr_256_nump_32
- mtllora_tr_128_nump_16

#### this is the latest version (17.07.2026) of the list of approaches we use for follow up training